# PPL Controller Interface
This notebook allows you to load a trained model and configure an MCTS-DMC controller to optimize process variables.

In [1]:
import sys
import os
import glob
import yaml
import pandas as pd
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))
from src.controller_logic import PPLController

%matplotlib inline

## Step 1: Select Model
Choose a previous training run to load config and LIME values.

In [ ]:
# List available runs
output_dir = '../output'
runs = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d))]
runs = sorted(runs, reverse=True)

# [Updated] Multi-Select Widget
model_selector = widgets.SelectMultiple(
    options=runs,
    description='Select Runs:',
    layout=widgets.Layout(width='400px', height='200px')
)

load_btn = widgets.Button(description="Load Models", button_style='primary')
load_output = widgets.Output()

controller = None
config = None

def on_load(b):
    global controller, config
    with load_output:
        clear_output()
        selected_runs = model_selector.value
        if not selected_runs:
            print("No runs selected.")
            return
            
        run_paths = [os.path.join(output_dir, r) for r in selected_runs]
        print(f"Loading from {len(run_paths)} runs...")
        
        try:
            # Initialize Controller with list of paths
            controller = PPLController(run_paths)
            config = controller.config
            print("Models Loaded successfully.")
            print(f"Inputs (MVs): {config['input_columns']}")
            print(f"Outputs (CVs): {config['output_columns']}")
            print(f"Total Context Samples: {len(controller.lime_df)}")
        except Exception as e:
            print(f"Error loading models: {e}")

load_btn.on_click(on_load)
display(model_selector, load_btn, load_output)

SelectMultiple(description='Select Runs:', layout=Layout(height='200px', width='400px'), options=('experiment_…

Button(button_style='primary', description='Load Models', style=ButtonStyle())

Output()

## Step 2: Configure Controller
Define limits for MVs, targets for CVs, and the Objective Function.

In [ ]:
config_output = widgets.Output()
config_ui_container = widgets.VBox()

def build_config_ui(b=None):
    global mv_widgets, cv_widgets, obj_widgets
    with config_output:
        clear_output()
        if controller is None:
            print("Please load a model first.")
            return
            
        # MV Config
        mv_label = widgets.HTML("<h3>Manipulated Variables (MVs)</h3>")
        mv_widgets = {}
        mv_items = []
        for mv in config['input_columns']:
            # Use default wide bounds for now
            w_min = widgets.FloatText(value=-10000.0, description='Min', layout=widgets.Layout(width='150px'))
            w_max = widgets.FloatText(value=10000.0, description='Max', layout=widgets.Layout(width='150px'))
            mv_widgets[mv] = {'min': w_min, 'max': w_max}
            mv_items.append(widgets.HBox([widgets.Label(mv, layout=widgets.Layout(width='200px')), w_min, w_max]))
            
        # CV Config
        cv_label = widgets.HTML("<h3>Controlled Variables (CVs)</h3>")
        cv_widgets = {}
        cv_items = []
        for cv in config['output_columns']:
            w_min = widgets.FloatText(value=-10000.0, description='Min', layout=widgets.Layout(width='120px'))
            w_max = widgets.FloatText(value=10000.0, description='Max', layout=widgets.Layout(width='120px'))
            w_target = widgets.FloatText(value=0.0, description='Target', layout=widgets.Layout(width='120px'))
            w_weight = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, description='Weight')
            
            cv_widgets[cv] = {'min': w_min, 'max': w_max, 'target': w_target, 'weight': w_weight}
            row = widgets.HBox([
                widgets.Label(cv, layout=widgets.Layout(width='150px')),
                w_min, w_max, w_target, w_weight
            ])
            cv_items.append(row)
            
        # Objective Function
        obj_label = widgets.HTML("<h3>Objective Function</h3>")
        obj_formula = widgets.Text(
            value="", 
            placeholder="e.g. (CV_1 * 10) - (MV_1 * 5)",
            description="Formula:",
            layout=widgets.Layout(width='500px')
        )
        obj_goal = widgets.Dropdown(options=['min', 'max'], value='min', description='Goal:')
        
        obj_widgets = {'formula': obj_formula, 'goal': obj_goal}
        
        # Apply Button
        apply_btn = widgets.Button(description="Apply Configuration", button_style='success')
        apply_btn.on_click(apply_config)
        
        config_ui_container.children = (
            mv_label, widgets.VBox(mv_items), 
            cv_label, widgets.VBox(cv_items),
            obj_label, widgets.HBox([obj_formula, obj_goal]),
            widgets.HTML("<hr>"),
            apply_btn
        )

def apply_config(b):
    # Parse widgets to dicts
    mvs_cfg = {k: {'min': v['min'].value, 'max': v['max'].value} for k, v in mv_widgets.items()}
    dvs_cfg = {} # TODO: Add DV support if DVs distinct from MVs in config
    cvs_cfg = {k: {
        'min': v['min'].value, 
        'max': v['max'].value, 
        'target': v['target'].value,
        'weight': v['weight'].value
    } for k, v in cv_widgets.items()}
    
    obj_f = obj_widgets['formula'].value
    obj_g = obj_widgets['goal'].value
    
    controller.configure(mvs_cfg, dvs_cfg, cvs_cfg, obj_f, obj_g)
    print("Configuration Applied.")
    

build_config_btn = widgets.Button(description="Build UI", button_style='info')
build_config_btn.on_click(build_config_ui)

display(build_config_btn, config_output, config_ui_container)

Button(button_style='info', description='Build UI', style=ButtonStyle())

Output()

VBox()

## Step 3: Run Controller
Execute the MCTS Controller for a single step.

In [ ]:
step_btn = widgets.Button(description="Run Step", button_style='danger')
step_output = widgets.Output()

# Mock initial state (random for prototype)
current_state = {}

def run_step(b):
    global current_state
    with step_output:
        if controller is None:
            print("Controller not configured.")
            return

        # Initialize state if empty
        if not current_state:
            # Start at 0 or mid-range of bounds
            for k in controller.mvs:
                current_state[k] = 0.0
            for k in controller.cvs:
                current_state[k] = 0.0
            print("Initialized state to zeros.")
            
        print(f"Current State: {current_state}")
        
        # Run MCTS
        print("Optimizing...")
        action = controller.search_mcts(current_state, iterations=50)
        print(f"Recommended Action: {action}")
        
        # Apply Action (Simulation)
        next_state = controller.step_dmc(current_state, action)
        current_state = next_state
        print(f"Next State: {current_state}")

step_btn.on_click(run_step)
display(step_btn, step_output)